# 03 — Mordred 2D Moleküler Özelliklerinin Üretilmesi

**Hedef:** Notebook 02 çıktısındaki parent SMILES'lardan Mordred 2D descriptorları üretmek, hatalı/eksik descriptorları temizlemek, yüksek korelasyonlu özellikleri azaltmak ve modellemeye hazır bir CSV hazırlamak.

### Bu notebookta öğreneceklerimiz
- Moleküler descriptor kavramı
- Mordred 2D descriptor hesaplama
- Sayısal dönüşüm ve sonsuz değer kontrolü
- Eksik değer, sabit özellik ve yüksek korelasyon filtresi
- Feature manifest ve tekrar üretilebilir çıktı

### Ders planındaki yeri
**Ders 2:** Moleküler temsil ve özellik mühendisliği.

## Workshop dosya yapısı

Bu seri çalıştırıldığında Google Drive altında aşağıdaki yapı oluşur:

```text
MyDrive/
└── CHEMBL206_workshop/
    ├── 01_data/
    ├── 02_cleaned/
    ├── 03_features/
    ├── 04_models/
    └── 05_reports/
```

> **Çalıştırma sırası:** Notebook 01 → 02 → 03 → 04 → 05  
> Her kod hücresinin sonunda bir **CHECKPOINT** mesajı vardır. Bir hücre hata verirse sonraki hücreye geçmeyiniz.

## Bilimsel not: Descriptor nedir?

Descriptor, bir molekülün yapısal veya fizikokimyasal bir özelliğini sayısal olarak temsil eder. Mordred 2D descriptorları yalnızca bağ grafiğinden hesaplanır; 3D konformer gerektirmez.

Bu notebookta üç temel temizlik uygulanır:

1. **Yüksek eksik oranı:** Değerlerinin %20'den fazlası eksik özellikler çıkarılır.
2. **Sabit özellik:** Tüm moleküllerde aynı olan özellikler çıkarılır.
3. **Yüksek korelasyon:** Mutlak Pearson korelasyonu 0.95'ten büyük özellik çiftlerinden biri çıkarılır.

> Eğitimsel sadeleştirme: Bu filtreler hedef değişken kullanılmadan yapılır. Çok katı bir validasyon çalışmasında tüm ön-işleme adımları yalnızca eğitim katında fit edilmelidir.

In [ ]:
# Google Drive bağlantısını kuruyoruz.
from google.colab import drive

# Drive'ı standart Colab yoluna bağlıyoruz.
drive.mount("/content/drive")

print("✅ CHECKPOINT 03.1: Google Drive bağlantısı kuruldu.")

In [ ]:
# Community-maintained Mordred sürümünü ve RDKit'i kuruyoruz.
# mordredcommunity paketi, kod içinde yine "mordred" adıyla içe aktarılır.
!pip -q install mordredcommunity==2.0.7 rdkit tqdm joblib

print("✅ CHECKPOINT 03.2: Mordred Community, RDKit ve yardımcı paketler kuruldu.")

In [ ]:
# Dosya yolları için pathlib kullanıyoruz.
from pathlib import Path

# Tablo işlemleri için pandas kullanıyoruz.
import pandas as pd

# Sayısal matris işlemleri için numpy kullanıyoruz.
import numpy as np

# JSON biçiminde feature manifest kaydetmek için json kullanıyoruz.
import json

# Python nesnelerini diske kaydetmek için joblib kullanıyoruz.
import joblib

# RDKit molekül okuma modülünü içe aktarıyoruz.
from rdkit import Chem

# Mordred hesaplayıcısını ve descriptor kataloğunu içe aktarıyoruz.
from mordred import Calculator, descriptors

# Eksik değer doldurma için SimpleImputer kullanıyoruz.
from sklearn.impute import SimpleImputer

# Notebook içinde tablo göstermek için display kullanıyoruz.
from IPython.display import display

# Proje klasörlerini tanımlıyoruz.
PROJECT_ROOT = Path("/content/drive/MyDrive/CHEMBL206_workshop")
CLEAN_DIR = PROJECT_ROOT / "02_cleaned"
FEATURE_DIR = PROJECT_ROOT / "03_features"

# Feature klasörünü oluşturuyoruz.
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Notebook 02 çıktısını giriş olarak tanımlıyoruz.
INPUT_CSV = CLEAN_DIR / "CHEMBL206_IC50_parent_dedup_Lipinski_filtered.csv"

# Descriptor temizleme eşiklerini tanımlıyoruz.
MAX_MISSING_RATIO = 0.20
CORRELATION_THRESHOLD = 0.95

# Giriş dosyasını doğruluyoruz.
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        "Notebook 02 çıktısı bulunamadı. Önce Notebook 02'yi çalıştırınız:\n"
        f"{INPUT_CSV}"
    )

print("Giriş CSV:", INPUT_CSV)
print("Feature klasörü:", FEATURE_DIR)
print("Eksik değer eşiği:", MAX_MISSING_RATIO)
print("Korelasyon eşiği:", CORRELATION_THRESHOLD)
print("✅ CHECKPOINT 03.3: Giriş dosyası ve feature ayarları hazır.")

In [ ]:
# Temizlenmiş parent veri setini okuyoruz.
df = pd.read_csv(INPUT_CSV, low_memory=False)

# Zorunlu sütunları tanımlıyoruz.
required_columns = {"parent_smiles", "pIC50"}

# Eksik zorunlu sütunları kontrol ediyoruz.
missing_columns = required_columns.difference(df.columns)

# Eksik sütun varsa işlemi durduruyoruz.
if missing_columns:
    raise KeyError(f"Eksik zorunlu sütunlar: {sorted(missing_columns)}")

# Parent SMILES'ları RDKit moleküllerine dönüştürüyoruz.
molecules = [Chem.MolFromSmiles(smiles) for smiles in df["parent_smiles"]]

# Geçersiz molekül sayısını hesaplıyoruz.
invalid_count = sum(mol is None for mol in molecules)

# Notebook 02'den sonra geçersiz yapı kalmaması gerekir.
assert invalid_count == 0, f"{invalid_count} SMILES RDKit tarafından okunamadı."

print("Molekül sayısı:", len(molecules))
print("Geçersiz molekül:", invalid_count)
display(df[["parent_smiles", "pIC50"]].head())

print("✅ CHECKPOINT 03.4: Moleküller RDKit nesnelerine dönüştürüldü.")

In [ ]:
# Yalnızca 2D descriptorları hesaplayan Mordred Calculator nesnesini oluşturuyoruz.
calculator = Calculator(
    descriptors,
    ignore_3D=True,
)

# Hesaplanacak descriptor sayısını yazdırıyoruz.
descriptor_count = len(calculator.descriptors)
print("Hesaplanacak 2D descriptor sayısı:", descriptor_count)

# Tüm moleküller için descriptorları hesaplıyoruz.
# nproc=1 workshop ortamında daha öngörülebilir bellek kullanımı sağlar.
mordred_raw = calculator.pandas(
    molecules,
    nproc=1,
    quiet=False,
)

# Descriptor adlarını güvenli metin sütunlarına çeviriyoruz.
mordred_raw.columns = [str(column) for column in mordred_raw.columns]

# Boyutu yazdırıyoruz.
print("Ham Mordred matrisi:", mordred_raw.shape)

assert mordred_raw.shape[0] == len(df), "Descriptor satır sayısı molekül sayısıyla uyuşmuyor."
assert mordred_raw.shape[1] > 0, "Hiç descriptor üretilemedi."

print("✅ CHECKPOINT 03.5: Ham Mordred 2D descriptorları üretildi.")

In [ ]:
# Mordred hata nesnelerini ve metinleri sayısal değere dönüştürmeyi deniyoruz.
# Dönüşmeyen değerler NaN olur.
mordred_numeric = mordred_raw.apply(
    pd.to_numeric,
    errors="coerce",
)

# Pozitif ve negatif sonsuz değerleri NaN'a çeviriyoruz.
mordred_numeric = mordred_numeric.replace(
    [np.inf, -np.inf],
    np.nan,
)

# Her descriptor için eksik değer oranını hesaplıyoruz.
missing_ratio = mordred_numeric.isna().mean()

# Eşik üzerindeki eksik oranına sahip descriptorları belirliyoruz.
high_missing_columns = missing_ratio[
    missing_ratio > MAX_MISSING_RATIO
].index.tolist()

# Yüksek eksikli descriptorları çıkarıyoruz.
X_step1 = mordred_numeric.drop(columns=high_missing_columns).copy()

print("Ham descriptor:", mordred_numeric.shape[1])
print("Yüksek eksik nedeniyle çıkarılan:", len(high_missing_columns))
print("Kalan descriptor:", X_step1.shape[1])

assert X_step1.shape[1] > 0, "Eksik değer filtresi sonrasında özellik kalmadı."

print("✅ CHECKPOINT 03.6: Sayısal dönüşüm ve eksik oran filtresi tamamlandı.")

In [ ]:
# Kalan eksik değerleri eğitimsel sadelik için descriptor medyanıyla dolduruyoruz.
imputer = SimpleImputer(strategy="median")

# İmputer'ı feature matrisi üzerinde fit edip dönüşüm uyguluyoruz.
X_imputed_array = imputer.fit_transform(X_step1)

# Numpy çıktısını aynı sütun adlarıyla DataFrame'e dönüştürüyoruz.
X_imputed = pd.DataFrame(
    X_imputed_array,
    columns=X_step1.columns,
    index=X_step1.index,
)

# Tüm satırlarda aynı değere sahip sabit descriptorları belirliyoruz.
constant_columns = [
    column
    for column in X_imputed.columns
    if X_imputed[column].nunique(dropna=False) <= 1
]

# Sabit descriptorları çıkarıyoruz.
X_step2 = X_imputed.drop(columns=constant_columns).copy()

print("Medyan doldurma sonrası eksik hücre:", int(X_imputed.isna().sum().sum()))
print("Sabit olduğu için çıkarılan:", len(constant_columns))
print("Kalan descriptor:", X_step2.shape[1])

assert not X_step2.isna().any().any(), "Feature matrisinde eksik değer kaldı."
assert X_step2.shape[1] > 0, "Sabit özellik filtresi sonrasında özellik kalmadı."

print("✅ CHECKPOINT 03.7: Medyan doldurma ve sabit özellik filtresi tamamlandı.")

In [ ]:
# Descriptorlar arasındaki mutlak Pearson korelasyon matrisini hesaplıyoruz.
correlation_matrix = X_step2.corr(method="pearson").abs()

# Matriste yalnızca üst üçgeni tutuyoruz; böylece her çifti bir kez inceliyoruz.
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1,
    ).astype(bool)
)

# Herhangi bir başka descriptorla eşik üzeri korelasyona sahip sütunları belirliyoruz.
high_correlation_columns = [
    column
    for column in upper_triangle.columns
    if (upper_triangle[column] > CORRELATION_THRESHOLD).any()
]

# Yüksek korelasyonlu descriptorlardan birer tanesini çıkarıyoruz.
X_final = X_step2.drop(columns=high_correlation_columns).copy()

print("Korelasyon öncesi descriptor:", X_step2.shape[1])
print("Yüksek korelasyon nedeniyle çıkarılan:", len(high_correlation_columns))
print("Final descriptor sayısı:", X_final.shape[1])

assert X_final.shape[1] > 0, "Korelasyon filtresi sonrasında özellik kalmadı."
assert not X_final.columns.duplicated().any(), "Tekrarlı feature adı bulundu."

print("✅ CHECKPOINT 03.8: Yüksek korelasyon filtresi tamamlandı.")

In [ ]:
# Modelleme dosyasında tutulacak açıklayıcı metadata sütunlarını belirliyoruz.
metadata_candidates = [
    "molecule_id",
    "parent_smiles",
    "smiles_raw_example",
    "pIC50",
    "n_records",
    "pIC50_std",
    "pIC50_range",
    "activity_conflict_flag",
]

# Veri setinde gerçekten bulunan metadata sütunlarını seçiyoruz.
metadata_columns = [
    column
    for column in metadata_candidates
    if column in df.columns
]

# Metadata ve final descriptor matrisini birleştiriyoruz.
model_ready_df = pd.concat(
    [
        df[metadata_columns].reset_index(drop=True),
        X_final.reset_index(drop=True),
    ],
    axis=1,
)

# Çıktı dosyalarının yollarını tanımlıyoruz.
RAW_OUTPUT = FEATURE_DIR / "CHEMBL206_Mordred2D_raw_numeric.csv"
MODEL_READY_OUTPUT = FEATURE_DIR / "CHEMBL206_Mordred2D_model_ready.csv"
MANIFEST_OUTPUT = FEATURE_DIR / "CHEMBL206_Mordred2D_feature_manifest.json"
IMPUTER_OUTPUT = FEATURE_DIR / "CHEMBL206_Mordred2D_imputer.joblib"

# Ham sayısal descriptor tablosunu kaydediyoruz.
mordred_numeric.to_csv(RAW_OUTPUT, index=False)

# Modellemeye hazır metadata + descriptor tablosunu kaydediyoruz.
model_ready_df.to_csv(MODEL_READY_OUTPUT, index=False)

# İmputer nesnesini tekrar kullanım için kaydediyoruz.
joblib.dump(imputer, IMPUTER_OUTPUT)

# Feature temizleme manifestini hazırlıyoruz.
manifest = {
    "input_csv": str(INPUT_CSV),
    "n_molecules": int(len(df)),
    "n_raw_descriptors": int(mordred_numeric.shape[1]),
    "max_missing_ratio": float(MAX_MISSING_RATIO),
    "correlation_threshold": float(CORRELATION_THRESHOLD),
    "dropped_high_missing": high_missing_columns,
    "dropped_constant": constant_columns,
    "dropped_high_correlation": high_correlation_columns,
    "final_feature_names": X_final.columns.tolist(),
    "n_final_features": int(X_final.shape[1]),
    "metadata_columns": metadata_columns,
}

# Manifesti JSON olarak kaydediyoruz.
with open(MANIFEST_OUTPUT, "w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

print("Ham descriptor dosyası:", RAW_OUTPUT)
print("Model-ready dosya:", MODEL_READY_OUTPUT)
print("Feature manifest:", MANIFEST_OUTPUT)
print("İmputer:", IMPUTER_OUTPUT)
print("Final tablo boyutu:", model_ready_df.shape)

# Kritik çıktıların oluştuğunu kontrol ediyoruz.
assert MODEL_READY_OUTPUT.exists()
assert MANIFEST_OUTPUT.exists()
assert len(model_ready_df) == len(df)

print("✅ CHECKPOINT 03.9: Mordred feature çıktıları başarıyla kaydedildi.")

In [ ]:
# Final feature matrisinin kısa kalite özetini oluşturuyoruz.
feature_summary = pd.DataFrame(
    {
        "metric": [
            "molecule_count",
            "raw_descriptor_count",
            "final_descriptor_count",
            "total_missing_cells",
            "minimum_feature_variance",
            "maximum_feature_variance",
        ],
        "value": [
            len(X_final),
            mordred_numeric.shape[1],
            X_final.shape[1],
            int(X_final.isna().sum().sum()),
            float(X_final.var().min()),
            float(X_final.var().max()),
        ],
    }
)

# Özeti ekranda gösteriyoruz.
display(feature_summary)

# Özeti CSV olarak kaydediyoruz.
SUMMARY_OUTPUT = FEATURE_DIR / "CHEMBL206_Mordred2D_QC_summary.csv"
feature_summary.to_csv(SUMMARY_OUTPUT, index=False)

print("Kalite özeti:", SUMMARY_OUTPUT)
print("✅ CHECKPOINT 03.10: Feature kalite kontrol özeti tamamlandı.")

## Ders sonu değerlendirme soruları

1. Descriptor ile fingerprint arasındaki temel fark nedir?
2. Neden descriptor hata nesnelerini `NaN` olarak dönüştürdük?
3. Yüksek korelasyonlu iki özelliğin ikisini birden tutmak hangi modellerde sorun yaratabilir?
4. Korelasyon filtresi neden hedef değişken `pIC50` kullanılmadan yapılmalıdır?

**Sonraki notebook:** LazyPredict ile regresyon modellerinin taranması ve final ağaç modelinin kaydedilmesi.